# Day 3 模块 5：特征重要性——谁更重要

模型训完后，老板常问：到底哪些因素更影响营收？


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day3_cafe_sales.csv'),
    Path('day3') / 'day3_cafe_sales.csv',
    Path('教学课程') / 'day3' / 'day3_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day3_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


In [ ]:
# 准备特征和目标（沿用 Day 2 的思路）
feature_cols = ['price', 'promotion', 'is_weekend', 'temp', 'quality', 'competitors', 'holiday', 'location']
# 天气变成数字分：晴最好，大雨最差
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df = df.copy()
df['weather_score'] = df['weather_label'].map(weather_score_map)

X = df[feature_cols + ['weather_score']].copy()
y = df['sales'].copy()

print('特征列:', list(X.columns))
print('样本数:', len(X))
X.head()


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('训练集行数:', len(X_train))
print('测试集行数:', len(X_test))


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
print('训练完成，测试 R²:', round(rf.score(X_test, y_test), 3))


## 1. 什么是特征重要性

随机森林可以给出每个特征的重要性分数。

先这样理解：

> 这个特征在很多棵树里，帮助把营收高低分清楚的贡献有多大。

### 原理（只记 3 步，不推公式）

树在每个节点选刀时，会让「乱度」降一点（回归里常是平方误差 / 不纯度下降）。

对某个特征，可以粗记：

```text
1. 在所有树、所有用到该特征的切分上，把「乱度下降量」累加起来
2. 所有特征的累加值再归一化（加起来大约为 1）
3. 得到 importance：谁累加贡献大，谁分数高
```

所以：

> **重要性 ≈ 这个特征在森林里，靠切分帮忙把误差/不纯度降下来的总贡献（归一化后）。**

数值越大，通常表示这个特征对**模型**更有用。

注意：这是“模型眼里的重要性”，**不等于**严格的业务因果关系。


## 2. 取出并排序


In [ ]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_,
}).sort_values('importance', ascending=False)

importance['importance'] = importance['importance'].round(4)
importance


## 3. 画一张简单条形图


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(8, 4))
plt.barh(importance['feature'], importance['importance'], color='#6f4e37')
plt.gca().invert_yaxis()
plt.title('随机森林特征重要性')
plt.xlabel('importance')
plt.tight_layout()
plt.show()


## 4. 用业务语言说一句

请观察排名靠前的特征，写一句给老板听的话。

例如：

- “模型里价格和天气评分更靠前，周末也有一定贡献。”
- 不要说绝对因果，可以说“模型更依赖……”


In [ ]:
# 写一句你的观察


## 5. 小练习

- 找出重要性最高的 3 个特征
- 重要性最低的特征是谁？你觉得合理吗？


In [ ]:
# 在这里写
